# Пайплайны обработки данных на основе LangChain и LLM

Ноутбук демонстрирует все части работы на модели qwen3.8-27b

Логика вынесена в модули src/*, офлайн-тесты - в tests/

In [1]:
import sys, json, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

from src.config import make_llm
from src.db import init_db, search_products, format_products, PRODUCTS
from src.schema import ProductQuery

In [2]:
init_db()
llm = make_llm(thinking=False)
RESULTS = {}

def md_table(headers, rows):
    esc = lambda v: str(v).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(esc(v) for v in r) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

def params(q):
    return "—" if q is None else f"{q.category}, {q.max_price}, {q.sort_by}"

def check(q, expected):
    """Сравнивает ProductQuery с ожиданием (category, max_price, sort_by)."""
    if q is None:
        return "Нет результата"
    got = (q.category, q.max_price, q.sort_by)
    bad = [n for n, g, e in zip(("category", "max_price", "sort_by"), got, expected) if g != e]
    return "Верно" if not bad else "Ошибка: " + ", ".join(bad)

print("Модель:", llm.model_name)

Модель: qwen3.8-27b


## Часть 1. Базовая цепочка LangChain

### Задание 1.1. Параметризованная цепочка
ChatPromptTemplate: роль модели в system, запрос в human через {query}, язык ответа {language}

In [3]:
from src.chains import category_prompt, build_category_chain

for m in category_prompt.format_messages(query="Нужен ноутбук для учебы", language="русский"):
    print(f"{m.type:>6}: {m.content}")

system: Ты - ассистент интернет-магазина электроники. Определи, к какой категории товаров относится запрос покупателя (например: ноутбуки, смартфоны, планшеты или другое). Ответь одной короткой фразой. Язык ответа: русский.
 human: Нужен ноутбук для учебы


In [4]:
messages = category_prompt.format_messages(query="Нужен ноутбук для учебы", language="русский")
direct = llm.invoke(messages)  # Вызов LLM
category_chain = build_category_chain(llm)  # prompt | llm | StrOutputParser
via_chain = category_chain.invoke({"query": "Нужен ноутбук для учебы", "language": "русский"})

print("llm.invoke   ->", type(direct).__name__, "| content:", repr(direct.content))
print("usage_metadata:", direct.usage_metadata)
print("chain.invoke ->", type(via_chain).__name__, "|", repr(via_chain))
RESULTS["part1_types"] = {"llm": type(direct).__name__, "chain": type(via_chain).__name__}

llm.invoke   -> AIMessage | content: 'Ноутбуки'
usage_metadata: {'input_tokens': 82, 'output_tokens': 4, 'total_tokens': 86, 'input_token_details': {}, 'output_token_details': {}}
chain.invoke -> TextAccessor | 'Ноутбуки'


Непосредственный вызов возвращает объект сообщения AIMessage, кроме текста в нем есть метаданные (usage_metadata с числом токенов, response_metadata). Цепочка с StrOutputParser возвращает только текст

In [5]:
batch_inputs = [
    {"query": "Посоветуй смартфон с хорошей камерой", "language": "русский"},
    {"query": "Ищу планшет для рисования", "language": "английский"},
    {"query": "Нужен холодильник No Frost", "language": "русский"},
    {"query": "Gaming laptop under 1000 dollars", "language": "русский"},
]
t0 = time.perf_counter()
answers = category_chain.batch(batch_inputs)
print(f"batch из {len(answers)} запросов: {time.perf_counter() - t0:.1f} c")
md_table(["Запрос", "Язык", "Ответ цепочки"],
         [(i["query"], i["language"], a) for i, a in zip(batch_inputs, answers)])
RESULTS["part1_batch"] = [[i["query"], i["language"], a] for i, a in zip(batch_inputs, answers)]

batch из 4 запросов: 60.2 c


| Запрос | Язык | Ответ цепочки |
|---|---|---|
| Посоветуй смартфон с хорошей камерой | русский | Смартфоны |
| Ищу планшет для рисования | английский | Tablets |
| Нужен холодильник No Frost | русский | Бытовая техника |
| Gaming laptop under 1000 dollars | русский | Ноутбуки |

batch обработал все четыре запроса одной командой. Переменная language работает, но для второго запроса ответ пришел на английском, а англоязычный запрос получил ответ на русском. Холодильник модель отнесла к бытовой технике - в части 1 категория свободная, перечисление появляется только в части 2

## Часть 2. Structured Output и Pydantic

### Задание 2.1. Схема структурированного ответа (src/schema.py)

Неизвестная категория -> other задается описанием поля и правилами в промпте. Cама схема other не подставляет, чтобы значения вне перечисления оставались ошибкой валидации

In [6]:
print(json.dumps(ProductQuery.model_json_schema(), ensure_ascii=False, indent=2))

{
  "properties": {
    "category": {
      "default": "other",
      "description": "Категория товара: laptop (ноутбуки), smartphone (смартфоны, телефоны), tablet (планшеты). Любая другая категория (холодильники, наушники и т.п.) или если категория не указана — other.",
      "enum": [
        "laptop",
        "smartphone",
        "tablet",
        "other"
      ],
      "title": "Category",
      "type": "string"
    },
    "max_price": {
      "anyOf": [
        {
          "minimum": 0,
          "type": "integer"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "description": "Верхняя граница цены в рублях, целое число без пробелов и единиц («до 50 тысяч» → 50000, «дешевле 1,5 млн» → 1500000). null, если ограничение цены не указано.",
      "title": "Max Price"
    },
    "sort_by": {
      "default": "price_asc",
      "description": "Сортировка по цене: price_asc — сначала дешевые (по умолчанию), price_desc — сначала дорогие («самые

### Задание 2.2. Инструкции по формату
PydanticOutputParser.get_format_instructions() подставляется в системный промпт через .partial(...)

In [7]:
from src.chains import product_parser, query_prompt, build_parse_chain

fi = product_parser.get_format_instructions()
print(fi[:400], "...\n")
sys_msg = query_prompt.format_messages(query="Найди ноутбуки")[0]
print("Инструкции формата внутри system-сообщения:", fi in sys_msg.content, "| роль:", sys_msg.type)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is n ...

Инструкции формата внутри system-сообщения: True | роль: system


In [8]:
parse_chain = build_parse_chain(llm)
q = parse_chain.invoke({"query": "Найди ноутбуки дешевле 70000 рублей"})
print(type(q).__name__, "->", q)
print("JSON:", q.model_dump_json())

ProductQuery -> category='laptop' max_price=70000 sort_by='price_asc'
JSON: {"category":"laptop","max_price":70000,"sort_by":"price_asc"}


Результат цепочки prompt | llm | PydanticOutputParser - уже проверенный объект ProductQuery, а не строка

model_dump_json() сериализует его в JSON для передачи дальше по пайплайну

### Задание 2.3. Тестирование

In [9]:
TESTS_23 = [
    ("Найди ноутбуки дешевле 70000 рублей", ("laptop", 70000, "price_asc"), "Категория и числовое ограничение"),
    ("Покажи самые дорогие смартфоны", ("smartphone", None, "price_desc"), "Сортировка по убыванию цены"),
    ("Хочу посмотреть планшеты", ("tablet", None, "price_asc"), "Отсутствие ограничения цены"),
    ("Покажи ноутбуки до 50 тысяч", ("laptop", 50000, "price_asc"), "Нормализация числового выражения"),
    ("Найди холодильники", ("other", None, "price_asc"), "Неизвестная категория other"),
]
rows, RESULTS["part2_tests"] = [], []
for text, exp, what in TESTS_23:
    q = parse_chain.invoke({"query": text})
    rows.append((text, q.model_dump_json(), what, check(q, exp)))
    RESULTS["part2_tests"].append([text, q.model_dump_json(), what, check(q, exp)])
md_table(["Запрос", "JSON", "Что проверяем", "Проверка полей"], rows)

| Запрос | JSON | Что проверяем | Проверка полей |
|---|---|---|---|
| Найди ноутбуки дешевле 70000 рублей | {"category":"laptop","max_price":70000,"sort_by":"price_asc"} | Категория и числовое ограничение | Верно |
| Покажи самые дорогие смартфоны | {"category":"smartphone","max_price":null,"sort_by":"price_desc"} | Сортировка по убыванию цены | Верно |
| Хочу посмотреть планшеты | {"category":"tablet","max_price":null,"sort_by":"price_asc"} | Отсутствие ограничения цены | Верно |
| Покажи ноутбуки до 50 тысяч | {"category":"laptop","max_price":50000,"sort_by":"price_asc"} | Нормализация числового выражения | Верно |
| Найди холодильники | {"category":"other","max_price":null,"sort_by":"price_asc"} | Неизвестная категория other | Верно |

Все пять контрольных запросов распознаны верно: 
* числовое ограничение («дешевле 70000»)
* сортировка по убыванию («самые дорогие»)
* отсутствие цены -> `null`
* нормализация «до 50 тысяч» -> `50000`
* неизвестная категория «холодильники» -> `other`

In [10]:
AMBIGUOUS = [
    "Подбери что-нибудь для учебы до 60к, лучше подороже",
    "Нужен iPad или MacBook, бюджет полтинник",
]
rows, RESULTS["part2_ambiguous"] = [], []
for text in AMBIGUOUS:
    q = parse_chain.invoke({"query": text})
    rows.append((text, q.model_dump_json()))
    RESULTS["part2_ambiguous"].append([text, q.model_dump_json()])
md_table(["Неоднозначный запрос", "JSON"], rows)

| Неоднозначный запрос | JSON |
|---|---|
| Подбери что-нибудь для учебы до 60к, лучше подороже | {"category":"other","max_price":60000,"sort_by":"price_desc"} |
| Нужен iPad или MacBook, бюджет полтинник | {"category":"other","max_price":50000,"sort_by":"price_asc"} |

В первом запросе категория не названа, поэтому other, «60к» преобразовано в 60000, «лучше подороже» понято как price_desc. Для пользователя это означает сообщение о неподдерживаемой категории, что разумнее, чем угадывать между ноутбуком и планшетом
Во втором же названы две категории сразу, а схема допускает одну - модель выбрала other. «Полтинник» корректно превращено в 50000. Такой запрос стоило бы уточнить у пользователя или разбить на два поиска - это ограничение схемы с одной категорией

### Задание 2.4. Режим размышления
Сравниваем одну и ту же цепочку с enable_thinking=False/True на двух неоднозначных запросах:

параметры, время, число сгенерированных токенов, содержимое content и работу парсера.

In [11]:
llm_think = make_llm(thinking=True)
raw_off, raw_on = query_prompt | llm, query_prompt | llm_think

def run_raw(chain, text):
    t0 = time.perf_counter()
    msg = chain.invoke({"query": text})
    dt = time.perf_counter() - t0
    try:
        parsed, ok = product_parser.invoke(msg).model_dump_json(), "Верно"
    except Exception as e:
        parsed, ok = type(e).__name__, "Ошибка"
    reasoning = msg.additional_kwargs.get("reasoning_content") or msg.additional_kwargs.get("reasoning") or ""
    return msg, dt, parsed, ok, reasoning

In [12]:
rows, RESULTS["part2_thinking"] = [], []
think_msgs = {}
for text in AMBIGUOUS:
    for mode, chain in (("off", raw_off), ("on", raw_on)):
        msg, dt, parsed, ok, reasoning = run_raw(chain, text)
        think_msgs[(text, mode)] = msg
        u = msg.usage_metadata or {}
        row = (text, mode, parsed, f"{dt:.1f}", u.get("output_tokens"),
               "<think>" in msg.content, len(reasoning), ok)
        rows.append(row); RESULTS["part2_thinking"].append(list(row))
md_table(["Запрос", "thinking", "Параметры", "Время, с", "output_tokens",
          "<think> в content", "длина reasoning вне content", "Парсер"], rows)

| Запрос | thinking | Параметры | Время, с | output_tokens | <think> в content | длина reasoning вне content | Парсер |
|---|---|---|---|---|---|---|---|
| Подбери что-нибудь для учебы до 60к, лучше подороже | off | {"category":"other","max_price":60000,"sort_by":"price_desc"} | 0.4 | 35 | False | 0 | Верно |
| Подбери что-нибудь для учебы до 60к, лучше подороже | on | {"category":"other","max_price":60000,"sort_by":"price_desc"} | 1.3 | 151 | False | 0 | Верно |
| Нужен iPad или MacBook, бюджет полтинник | off | {"category":"other","max_price":50000,"sort_by":"price_asc"} | 0.4 | 35 | False | 0 | Верно |
| Нужен iPad или MacBook, бюджет полтинник | on | {"category":"tablet","max_price":50000,"sort_by":"price_asc"} | 12.3 | 1185 | False | 0 | Верно |

In [13]:
msg = think_msgs[(AMBIGUOUS[0], "on")]
print("content (thinking=on):", repr(msg.content[:300]))
print("ключи additional_kwargs:", list(msg.additional_kwargs))
print("output_token_details:", (msg.usage_metadata or {}).get("output_token_details"))

content (thinking=on): '\n\n{"category":"other","max_price":60000,"sort_by":"price_desc"}'
ключи additional_kwargs: ['refusal']
output_token_details: {}


То же самое с возвратом рассуждения

In [14]:
import os
from openai import OpenAI
from src.config import get_api_key, DEFAULT_BASE_URL, DEFAULT_MODEL

raw_client = OpenAI(base_url=os.getenv("LITELLM_BASE_URL", DEFAULT_BASE_URL), api_key=get_api_key())
resp = raw_client.chat.completions.create(
    model=os.getenv("MODEL_NAME", DEFAULT_MODEL),
    messages=[{"role": m.type.replace("human", "user"), "content": m.content}
              for m in query_prompt.format_messages(query=AMBIGUOUS[0])],
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
    temperature=0,
)
m = resp.choices[0].message.model_dump()
for k in ("reasoning_content", "reasoning"):
    if m.get(k):
        print(f"message.{k}:", repr(m[k][:300]), "...")
print("message.content:", repr(m["content"][:200]))
print("usage:", resp.usage.model_dump())
RESULTS["part2_raw_fields"] = [k for k, v in m.items() if v]

message.reasoning_content: 'Нужно извлечь параметры из запроса: "Подбери что-нибудь для учебы до 60к, лучше подороже". Категория не названа явно, "что-нибудь для учебы" — other. max_price: до 60к = 60000. sort_by: "лучше подороже" означает сначала дорогие → price_desc. Ответ только JSON без markdown.\n' ...
message.content: '\n\n{"category":"other","max_price":60000,"sort_by":"price_desc"}'
usage: {'completion_tokens': 111, 'prompt_tokens': 712, 'total_tokens': 823, 'completion_tokens_details': None, 'prompt_tokens_details': None}


* Параметры. На первом запросе режимы дали одинаковый результат. На втором («iPad или MacBook») без размышления
  модель выбрала other (две категории сразу), а с размышлением - tablet (первый названный товар). Ни одна
  интерпретация не правильнее - запрос неоднозначен, и размышление лишь выбрало другой вариант. Режим размышления
  к тому же менее стабилен, в предыдущем прогоне на этом же запросе он дал other, потратив 6163 токена и 58 с.
* Без размышления ответ занимает 35 токенов и меньше одной секнды; с ним - 151 токен на первом запросе и
  1185 токенов 12 секунд на втором. Для извлечения параметров это большие затраты без явного выигрыша в качестве
* Сервер (vLLM с reasoning-парсером) отдает рассуждение в отдельном поле message.reasoning_content
  (видно в сыром ответе OpenAI-клиента), а в content остается только JSON. Тегов <think>
  в content нет, поэтому PydanticOutputParser справился в обоих режимах. ChatOpenAI поле reasoning_content
  в additional_kwargs не переносит, поэтому в таблице его длина 0

Почему рассуждение не должно попадать в разбираемый текст - это свободный текст с кавычками, фигурными скобками и черновыми вариантами JSON; парсер либо упадет, либо вытащит промежуточный вариант вместо итогового, что хуже. Если бы сервер не отделял рассуждение, его пришлось бы вырезать до парсинга

## Часть 3. Вызов классического программного обеспечения

### Задание 3.1. Каталог товаров (src/db.py, scripts/init_db.py)

In [15]:
md_table(["id", "Название", "Категория", "Цена, ₽"],
         [(i + 1, n, c, f"{p:,}".replace(",", " ")) for i, (n, c, p) in enumerate(PRODUCTS)])

| id | Название | Категория | Цена, ₽ |
|---|---|---|---|
| 1 | HONOR MagicBook X16 AMD 2025 | laptop | 70 299 |
| 2 | MSI Katana 17 B13UCR-1491XRU | laptop | 99 299 |
| 3 | Tecno Megabook K15SDA | laptop | 55 299 |
| 4 | DEXP Atlas A15-I3W400 | laptop | 44 299 |
| 5 | ASUS TUF Gaming FX707VJB-HX098 | laptop | 97 299 |
| 6 | Apple MacBook Air M5 | laptop | 143 999 |
| 7 | Apple iPhone 17 | smartphone | 88 790 |
| 8 | Xiaomi REDMI Note 15 Pro | smartphone | 24 299 |
| 9 | HONOR 600 | smartphone | 59 999 |
| 10 | Samsung Galaxy A57 | smartphone | 41 999 |
| 11 | POCO X8 Pro Max | smartphone | 58 799 |
| 12 | Apple iPad (11th Gen) | tablet | 45 999 |
| 13 | Xiaomi REDMI Pad 2 LTE | tablet | 27 999 |
| 14 | HUAWEI MatePad SE 11 Wi-Fi | tablet | 24 999 |
| 15 | HONOR Pad X9a LTE | tablet | 30 999 |
| 16 | SteelSeries Arctis Nova 1 | other | 7 299 |

### Задание 3.2. Программный поиск
SQL строится кодом из белого списка полей, значения передаются параметрами ?

Поведение для category = other - поиск не выполняется, пользователю возвращается сообщение о неподдерживаемой категории, чтобы не выводить весь католог

In [16]:
from src.db import build_sql, UnsupportedCategoryError

q = ProductQuery(category="laptop", max_price=70000)
print(build_sql(q))
print(format_products(search_products(q)))
print(format_products(search_products(ProductQuery(category="smartphone", sort_by="price_desc"))))
try:
    search_products(ProductQuery(category="other"))
except UnsupportedCategoryError as e:
    print("other ->", type(e).__name__)

('SELECT id, name, category, price FROM products WHERE category = ? AND price <= ? ORDER BY price ASC, id', ['laptop', 70000])
DEXP Atlas A15-I3W400 (44 299 ₽); Tecno Megabook K15SDA (55 299 ₽)
Apple iPhone 17 (88 790 ₽); HONOR 600 (59 999 ₽); POCO X8 Pro Max (58 799 ₽); Samsung Galaxy A57 (41 999 ₽); Xiaomi REDMI Note 15 Pro (24 299 ₽)
other -> UnsupportedCategoryError


### Задание 3.3. Объединение компонентов
Функция поиска включена в LCEL через RunnableLambda: prompt | llm | parser | RunnableLambda(run_search)

In [17]:
from src.chains import build_search_chain

search_chain = build_search_chain(llm)
res = search_chain.invoke({"query": "Покажи ноутбуки до 80 тысяч"})
print(res["status"], "|", res["query"], "|", format_products(res["products"]))

ok | category='laptop' max_price=80000 sort_by='price_asc' | DEXP Atlas A15-I3W400 (44 299 ₽); Tecno Megabook K15SDA (55 299 ₽); HONOR MagicBook X16 AMD 2025 (70 299 ₽)


In [18]:
TESTS_33 = [
    ("Найди ноутбуки дешевле 70000 рублей", ("laptop", 70000, "price_asc")),
    ("Покажи самые дорогие смартфоны", ("smartphone", None, "price_desc")),
    ("Хочу посмотреть планшеты", ("tablet", None, "price_asc")),
    ("Покажи ноутбуки до 50 тысяч", ("laptop", 50000, "price_asc")),
    ("Найди холодильники", ("other", None, "price_asc")),
    ("Смартфоны не дороже 30к", ("smartphone", 30000, "price_asc")),
    ("Есть планшет тысяч за пятнадцать?", ("tablet", 15000, "price_asc")),
    ("Самые дорогие ноутбуки до 1,5 млн", ("laptop", 1500000, "price_desc")),
    ("Какой-нибудь бюджетный телефон до десяти тысяч", ("smartphone", 10000, "price_asc")),
    ("Покажи планшеты подороже, но не больше 100 тысяч", ("tablet", 100000, "price_desc")),
    ("Нужны наушники до 40 тысяч", ("other", 40000, "price_asc")),
    ("Ноутбук за 30 тысяч есть?", ("laptop", 30000, "price_asc")),
]
rows, RESULTS["part3_table"] = [], []
for text, exp in TESTS_33:
    r = search_chain.invoke({"query": text})
    found = format_products(r["products"]) if r["status"] != "unsupported" else r["message"]
    row = (text, params(r["query"]), found, check(r["query"], exp))
    rows.append(row); RESULTS["part3_table"].append(list(row))
md_table(["Запрос пользователя", "Структурированные параметры", "Найденные товары", "Корректность"], rows)
ok = sum(r[3] == "Верно" for r in rows)
print(f"Корректно: {ok}/{len(rows)}")

| Запрос пользователя | Структурированные параметры | Найденные товары | Корректность |
|---|---|---|---|
| Найди ноутбуки дешевле 70000 рублей | laptop, 70000, price_asc | DEXP Atlas A15-I3W400 (44 299 ₽); Tecno Megabook K15SDA (55 299 ₽) | Верно |
| Покажи самые дорогие смартфоны | smartphone, None, price_desc | Apple iPhone 17 (88 790 ₽); HONOR 600 (59 999 ₽); POCO X8 Pro Max (58 799 ₽); Samsung Galaxy A57 (41 999 ₽); Xiaomi REDMI Note 15 Pro (24 299 ₽) | Верно |
| Хочу посмотреть планшеты | tablet, None, price_asc | HUAWEI MatePad SE 11 Wi-Fi (24 999 ₽); Xiaomi REDMI Pad 2 LTE (27 999 ₽); HONOR Pad X9a LTE (30 999 ₽); Apple iPad (11th Gen) (45 999 ₽) | Верно |
| Покажи ноутбуки до 50 тысяч | laptop, 50000, price_asc | DEXP Atlas A15-I3W400 (44 299 ₽) | Верно |
| Найди холодильники | other, None, price_asc | Категория не поддерживается: ищем только ноутбуки, смартфоны и планшеты. | Верно |
| Смартфоны не дороже 30к | smartphone, 30000, price_asc | Xiaomi REDMI Note 15 Pro (24 299 ₽) | Верно |
| Есть планшет тысяч за пятнадцать? | tablet, 15000, price_asc | — | Верно |
| Самые дорогие ноутбуки до 1,5 млн | laptop, 1500000, price_desc | Apple MacBook Air M5 (143 999 ₽); MSI Katana 17 B13UCR-1491XRU (99 299 ₽); ASUS TUF Gaming FX707VJB-HX098 (97 299 ₽); HONOR MagicBook X16 AMD 2025 (70 299 ₽); Tecno Megabook K15SDA (55 299 ₽); DEXP Atlas A15-I3W400 (44 299 ₽) | Верно |
| Какой-нибудь бюджетный телефон до десяти тысяч | smartphone, 10000, price_asc | — | Верно |
| Покажи планшеты подороже, но не больше 100 тысяч | tablet, 100000, price_desc | Apple iPad (11th Gen) (45 999 ₽); HONOR Pad X9a LTE (30 999 ₽); Xiaomi REDMI Pad 2 LTE (27 999 ₽); HUAWEI MatePad SE 11 Wi-Fi (24 999 ₽) | Верно |
| Нужны наушники до 40 тысяч | other, 40000, price_asc | Категория не поддерживается: ищем только ноутбуки, смартфоны и планшеты. | Верно |
| Ноутбук за 30 тысяч есть? | laptop, 30000, price_asc | — | Верно |

Корректно: 12/12


12 из 12 запросов преобразованы в верные параметры. В выборке есть все исходы: несколько товаров или один товар, пустой результат (планшет за 15 тысяч, телефон до 10 тысяч, ноутбук за 30 тысяч) и неподдерживаемая категория. Сортировка price_desc и большие числа (1,5 млн -> 1500000) отработаны корректно

#### Модель без доступа к базе

In [19]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

no_db = ChatPromptTemplate.from_messages([
    ("system", "Ты — консультант интернет-магазина электроники. Отвечай кратко."),
    ("human", "{q}"),
]) | llm | StrOutputParser()

NO_DB = [
    ("Какие ноутбуки дешевле 70000 рублей есть в каталоге?", ProductQuery(category="laptop", max_price=70000)),
    ("Какие самые дорогие смартфоны есть в каталоге?", ProductQuery(category="smartphone", sort_by="price_desc")),
    ("Есть ли в каталоге планшеты до 15000 рублей?", ProductQuery(category="tablet", max_price=15000)),
]
RESULTS["part3_no_db"] = []
for question, q in NO_DB:
    answer = no_db.invoke({"q": question})
    real = format_products(search_products(q))
    print(f"### {question}\nМодель без базы: {answer.strip()[:600]}\nПоиск по базе: {real}\n")
    RESULTS["part3_no_db"].append([question, answer.strip(), real])

### Какие ноутбуки дешевле 70000 рублей есть в каталоге?
Модель без базы: У меня нет доступа к вашему текущему каталогу или базе данных товаров.

Пожалуйста, уточните:
1. Название магазина или ссылку на каталог.
2. Или пришлите список моделей с ценами, которые вы видите.

Тогда я смогу отфильтровать варианты дешевле 70 000 рублей.
Поиск по базе: DEXP Atlas A15-I3W400 (44 299 ₽); Tecno Megabook K15SDA (55 299 ₽)

### Какие самые дорогие смартфоны есть в каталоге?
Модель без базы: У меня нет доступа к вашему текущему каталогу или базе данных магазина.

Пожалуйста, уточните:
1. Название магазина или ссылку на каталог.
2. Либо пришлите список моделей, которые вас интересуют.

Тогда я смогу помочь с выбором.
Поиск по базе: Apple iPhone 17 (88 790 ₽); HONOR 600 (59 999 ₽); POCO X8 Pro Max (58 799 ₽); Samsung Galaxy A57 (41 999 ₽); Xiaomi REDMI Note 15 Pro (24 299 ₽)

### Есть ли в каталоге планшеты до 15000 рублей?
Модель без базы: Я не имею доступа к вашему текущему каталогу и актуальным це

Без доступа к каталогу модель честно отвечает, что не знает ассортимента, и просит ссылку на магазин или список товаров

В одном из прогонов модель дополнительно перечислила примеры для выборп (iPhone Pro Max, Galaxy S Ultra), но их в нашем каталоге нет

Итог: поиск по базе дает точный и проверяемый список (или подтверждает, что планшетов до 15 000 ₽ нет). Модель не знает содержимого конкретной базы, поэтому фактические данные должна возвращать программа, а LLM - только интерпретировать запрос

### Задание 3.4. Поиск как инструмент
@tool("search_products", args_schema=ProductQuery) + bind_tools. Приложение валидирует tool_calls[0].args по ProductQuery и только потом ищет, результат модели не возвращается

In [20]:
from src.tools import search_products_tool, run_tool_query
from src.journal import Journal

print(search_products_tool.name, "-", search_products_tool.description)
print(json.dumps(search_products_tool.tool_call_schema.model_json_schema(), ensure_ascii=False)[:400])

search_products - Поиск товаров в каталоге интернет-магазина электроники по категории (laptop, smartphone, tablet или other), максимальной цене в рублях и сортировке по цене. Вызывай для любого запроса покупателя о товарах.
{"description": "Поиск товаров в каталоге интернет-магазина электроники по категории (laptop, smartphone, tablet или other), максимальной цене в рублях и сортировке по цене. Вызывай для любого запроса покупателя о товарах.", "properties": {"category": {"default": "other", "description": "Категория товара: laptop (ноутбуки), smartphone (смартфоны, телефоны), tablet (планшеты). Любая другая категори


In [21]:
rows, RESULTS["part3_tools"] = [], []
tool_queries = [(t, e) for t, e, _ in TESTS_23] + [("Привет! Расскажи анекдот", None)]
for text, exp in tool_queries:
    r = run_tool_query(llm, text)
    row = (text, repr(r.content[:60]), json.dumps(r.tool_calls, ensure_ascii=False), r.status,
           format_products(r.products) if r.status in ("ok", "empty") else r.message,
           check(r.query, exp) if exp else ("Верно (ожидался ответ без вызова)" if r.status == "no_tool_call" else "Ошибка"))
    rows.append(row); RESULTS["part3_tools"].append(list(row))
md_table(["Запрос", "content", "tool_calls", "Статус", "Результат", "Проверка"], rows)

| Запрос | content | tool_calls | Статус | Результат | Проверка |
|---|---|---|---|---|---|
| Найди ноутбуки дешевле 70000 рублей | '' | [{"name": "search_products", "args": {"category": "laptop", "max_price": 70000}}] | ok | DEXP Atlas A15-I3W400 (44 299 ₽); Tecno Megabook K15SDA (55 299 ₽) | Верно |
| Покажи самые дорогие смартфоны | '' | [{"name": "search_products", "args": {"category": "smartphone", "sort_by": "price_desc"}}] | ok | Apple iPhone 17 (88 790 ₽); HONOR 600 (59 999 ₽); POCO X8 Pro Max (58 799 ₽); Samsung Galaxy A57 (41 999 ₽); Xiaomi REDMI Note 15 Pro (24 299 ₽) | Верно |
| Хочу посмотреть планшеты | '' | [{"name": "search_products", "args": {"category": "tablet"}}] | ok | HUAWEI MatePad SE 11 Wi-Fi (24 999 ₽); Xiaomi REDMI Pad 2 LTE (27 999 ₽); HONOR Pad X9a LTE (30 999 ₽); Apple iPad (11th Gen) (45 999 ₽) | Верно |
| Покажи ноутбуки до 50 тысяч | '' | [{"name": "search_products", "args": {"category": "laptop", "max_price": 50000}}] | ok | DEXP Atlas A15-I3W400 (44 299 ₽) | Верно |
| Найди холодильники | '' | [{"name": "search_products", "args": {"category": "other"}}] | unsupported | Категория не поддерживается: ищем только ноутбуки, смартфоны и планшеты. | Верно |
| Привет! Расскажи анекдот | 'Привет! 😊\n\nВот один для тебя:\n\n— Доктор, я, кажется, стал не' | [] | no_tool_call | Модель не вызвала инструмент поиска. | Верно (ожидался ответ без вызова) |

In [22]:
print("Журнал для запроса без вызова инструмента:")
_ = run_tool_query(llm, "Привет! Расскажи анекдот", journal=Journal(echo=True))
print("\nЖурнал для запроса с вызовом инструмента:")
_ = run_tool_query(llm, "Покажи ноутбуки до 50 тысяч", journal=Journal(echo=True))

Журнал для запроса без вызова инструмента:
[  0.000s] request        query="Привет! Расскажи анекдот"
[  0.000s] llm_call       attempt=1 mode="bind_tools"
[  0.973s] tool_calls     n=0 calls=[]
[  0.974s] search_skipped reason="no_tool_call"

Журнал для запроса с вызовом инструмента:
[  0.000s] request        query="Покажи ноутбуки до 50 тысяч"
[  0.000s] llm_call       attempt=1 mode="bind_tools"
[  0.475s] tool_calls     n=1 calls=[{"name": "search_products", "args": {"category": "laptop", "max_price": 50000}}]
[  0.476s] validate_ok    params={"category": "laptop", "max_price": 50000, "sort_by": "price_asc"}
[  0.476s] search_start   params={"category": "laptop", "max_price": 50000, "sort_by": "price_asc"}
[  0.477s] search_done    found=1


Для всех пяти запросов из 2.3 модель вызвала search_products с верными аргументами, а content при этом пустой - параметры пришли в структурированном поле tool_calls, а не в тексте
Необязательные аргументы модель просто опускает, их подставляют значения по умолчанию ProductQuery. На запрос об анекдоте инструмент не вызван

| | Часть 2: инструкции формата | Часть 3.4: вызов инструмента |
|---|---|---|
| Где описана схема | В тексте системного промпта (get_format_instructions()) | В описании инструмента (JSON Schema в поле tools запроса) |
| Кто извлекает параметры из текста модели | PydanticOutputParser на стороне приложения разбирает JSON из content | Парсер вызовов на сервере (vLLM tool parser) превращает вывод модели в tool_calls |
| Кто проверяет | Pydantic внутри парсера | Приложение: ProductQuery.model_validate(args) |
| Кто выполняет действие | Программа (RunnableLambda(run_search)) | Программа, модель только предлагает вызов, результат ей не возвращается |
| Что если модель «не послушалась» | OutputParserException -> retry | Пустые tool_calls -> отдельная ветка «инструмент не вызван» |

## Часть 4. Retry и журнал выполнения

### Задание 4.1. Анализ ошибок

In [23]:
INVALID = [
    ("Синтаксически некорректный JSON", "{category: laptop, max_price: 70000}"),
    ("Текст вместо JSON", "Конечно! Вот ноутбуки до 70000 ₽: Acer Aspire 3"),
    ("Значение вне перечисления", '{"category": "phone", "max_price": 30000, "sort_by": "price_asc"}'),
    ("Неверный тип", '{"category": "laptop", "max_price": "дорого", "sort_by": "price_asc"}'),
    ("Нарушение ограничения ge=0", '{"category": "laptop", "max_price": -5, "sort_by": "price_asc"}'),
    ("Дробная цена", '{"category": "laptop", "max_price": 49999.5, "sort_by": "price_asc"}'),
]
LENIENT = [
    ("Незакрытая скобка", '{"category": "laptop", "max_price": 70000'),
    ("Число строкой", '{"category": "laptop", "max_price": "50000"}'),
    ("JSON в markdown, 50000.0", '```json\n{"category": "laptop", "max_price": 50000.0}\n```'),
]
rows, RESULTS["part4_errors"] = [], []

In [24]:
for name, raw in INVALID + LENIENT:
    try:
        rows.append((name, raw, "нет ошибки → " + product_parser.parse(raw).model_dump_json(), "—", "—"))
    except Exception as e:
        cause = e.__cause__
        detail = str(cause).splitlines()[1:3] if cause and type(cause).__name__ == "ValidationError" else [str(cause)]
        rows.append((name, raw, type(e).__name__, type(cause).__name__, " ".join(detail)[:120]))
    RESULTS["part4_errors"].append(list(rows[-1]))
md_table(["Случай", "Ответ модели", "Исключение", "__cause__", "Детали"], rows)

| Случай | Ответ модели | Исключение | __cause__ | Детали |
|---|---|---|---|---|
| Синтаксически некорректный JSON | {category: laptop, max_price: 70000} | OutputParserException | JSONDecodeError | Expecting property name enclosed in double quotes: line 1 column 2 (char 1) |
| Текст вместо JSON | Конечно! Вот ноутбуки до 70000 ₽: Acer Aspire 3 | OutputParserException | JSONDecodeError | Expecting value: line 1 column 1 (char 0) |
| Значение вне перечисления | {"category": "phone", "max_price": 30000, "sort_by": "price_asc"} | OutputParserException | ValidationError | category   Input should be 'laptop', 'smartphone', 'tablet' or 'other' [type=literal_error, input_value='phone', input_t |
| Неверный тип | {"category": "laptop", "max_price": "дорого", "sort_by": "price_asc"} | OutputParserException | ValidationError | max_price   Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='дорого |
| Нарушение ограничения ge=0 | {"category": "laptop", "max_price": -5, "sort_by": "price_asc"} | OutputParserException | ValidationError | max_price   Input should be greater than or equal to 0 [type=greater_than_equal, input_value=-5, input_type=int] |
| Дробная цена | {"category": "laptop", "max_price": 49999.5, "sort_by": "price_asc"} | OutputParserException | ValidationError | max_price   Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=49999 |
| Незакрытая скобка | {"category": "laptop", "max_price": 70000 | нет ошибки → {"category":"laptop","max_price":70000,"sort_by":"price_asc"} | — | — |
| Число строкой | {"category": "laptop", "max_price": "50000"} | нет ошибки → {"category":"laptop","max_price":50000,"sort_by":"price_asc"} | — | — |
| JSON в markdown, 50000.0 | ```json {"category": "laptop", "max_price": 50000.0} ``` | нет ошибки → {"category":"laptop","max_price":50000,"sort_by":"price_asc"} | — | — |

Все ошибки приходят как OutputParserException, а настоящая причина - в __cause__:
* JSONDecodeError - этап разбора JSON (синтаксис нарушен или вместо JSON обычный текст)
* ValidationError - этап валидации Pydantic - значение вне Literal (literal_error), неверный тип
  (int_parsing), нарушение ge=0 (greater_than_equal), дробная цена (int_from_float)

Последние три строки показывают снисходительность. Незакрытую скобку парсер достраивает, строку "50000" и
50000.0 Pydantic приводит к int, markdown-обертку снимает. Такие ответы проходят без ошибки, поэтому для
демонстрации retry выбраны только примеры, которые действительно вызывают исключение

### Задания 4.2-4.3. Retry и журнал
ProductSearchPipeline (src/pipeline.py): RunnableLambda(generate_and_parse).with_retry(stop_after_attempt=3,
retry_if_exception_type=(OutputParserException,)) -> validate -> search. Поиск стоит после границы retry

In [25]:
from src.pipeline import ProductSearchPipeline

journal = Journal(echo=True)
pipe = ProductSearchPipeline(llm, journal=journal)
r = pipe.invoke("Найди ноутбуки дешевле 70000 рублей")
print("\n->", r.status, "|", r.message, "|", format_products(r.products))

[  0.004s] request        query="Найди ноутбуки дешевле 70000 рублей"
[  0.005s] llm_call       attempt=1
[  0.430s] parse_ok       attempt=1 params={"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}
[  0.433s] validate_ok    category="laptop"
[  0.433s] search_start   params={"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}
[  0.435s] search_done    found=2
[  0.435s] result         status="ok" found=2

-> ok | Найдено товаров: 2 | DEXP Atlas A15-I3W400 (44 299 ₽); Tecno Megabook K15SDA (55 299 ₽)


In [26]:
# Дополнительно: подтверждение перед поиском без ограничения цены
def confirm(q):
    print(f"   ? Искать {q.category} без ограничения цены? -> да (автоответ в ноутбуке)")
    return True

pipe_c = ProductSearchPipeline(llm, journal=Journal(echo=True), confirm=confirm)
r = pipe_c.invoke("Покажи самые дорогие смартфоны")
print("\n->", r.status, "|", format_products(r.products))
pipe_c2 = ProductSearchPipeline(llm, journal=Journal(echo=True), confirm=lambda q: False)
print("\n->", pipe_c2.invoke("Хочу посмотреть планшеты").message)

[  0.000s] request        query="Покажи самые дорогие смартфоны"
[  0.001s] llm_call       attempt=1
[  0.381s] parse_ok       attempt=1 params={"category": "smartphone", "max_price": null, "sort_by": "price_desc"}
[  0.383s] validate_ok    category="smartphone"
   ? Искать smartphone без ограничения цены? -> да (автоответ в ноутбуке)
[  0.384s] confirm        question="поиск без ограничения цены" answer=True
[  0.384s] search_start   params={"category": "smartphone", "max_price": null, "sort_by": "price_desc"}
[  0.385s] search_done    found=5
[  0.385s] result         status="ok" found=5

-> ok | Apple iPhone 17 (88 790 ₽); HONOR 600 (59 999 ₽); POCO X8 Pro Max (58 799 ₽); Samsung Galaxy A57 (41 999 ₽); Xiaomi REDMI Note 15 Pro (24 299 ₽)
[  0.000s] request        query="Хочу посмотреть планшеты"
[  0.001s] llm_call       attempt=1
[  0.407s] parse_ok       attempt=1 params={"category": "tablet", "max_price": null, "sort_by": "price_asc"}
[  0.409s] validate_ok    category="tablet"
[

Журнал показывает все требуемые шаги: llm_call с номером попытки, parse_ok/parse_error, результат проверки validate_ok и search_start -> search_done с числом товаров

С реальной моделью ответ разобрался с первой попытки

Дополнительно - перед поиском без ограничения цены вызывается подтверждение, при отказе (answer=False) поиск не выполняется (status="cancelled")

### Задание 4.4. Воспроизводимый тест
FakeListChatModel возвращает заранее заданные ответы по очереди, функция поиска подменена счетчиком вызовов

In [27]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

BAD = "Конечно! Вот что я нашел: {category: laptop}"
GOOD = '{"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}'

class SpySearch:
    def __init__(self): self.calls = 0
    def __call__(self, q):
        self.calls += 1
        return search_products(q)

RESULTS["part4_fake"] = {}
for title, responses, use_retry in [
    ("A. Без retry: плохой → хороший", [BAD, GOOD], False),
    ("B. С retry: плохой → хороший", [BAD, GOOD], True),
    ("C. С retry: плохой ×3", [BAD, BAD, BAD], True),
]:
    print(f"\n{title}")
    spy, j = SpySearch(), Journal(echo=True)
    r = ProductSearchPipeline(FakeListChatModel(responses=responses), journal=j,
                              use_retry=use_retry, search_fn=spy).invoke("Найди ноутбуки дешевле 70000 рублей")
    print(f"-> status={r.status}, attempts={r.attempts}, вызовов поиска={spy.calls}, сообщение: {r.message}")
    RESULTS["part4_fake"][title] = {"status": r.status, "attempts": r.attempts,
                                    "search_calls": spy.calls, "journal": j.text()}


A. Без retry: плохой → хороший
[  0.000s] request        query="Найди ноутбуки дешевле 70000 рублей"
[  0.000s] llm_call       attempt=1
[  0.002s] parse_error    attempt=1 error="JSONDecodeError" raw="Конечно! Вот что я нашел: {category: laptop}"
[  0.002s] failed         reason="parse_error" attempts=1
[  0.002s] result         status="error" found=0
-> status=error, attempts=1, вызовов поиска=0, сообщение: Не удалось понять запрос: модель 1 раз(а) вернула ответ в неверном формате. Попробуйте переформулировать.

B. С retry: плохой → хороший
[  0.000s] request        query="Найди ноутбуки дешевле 70000 рублей"
[  0.000s] llm_call       attempt=1
[  0.001s] parse_error    attempt=1 error="JSONDecodeError" raw="Конечно! Вот что я нашел: {category: laptop}"
[  0.001s] llm_call       attempt=2
[  0.002s] parse_ok       attempt=2 params={"category": "laptop", "max_price": 70000, "sort_by": "price_asc"}
[  0.002s] validate_ok    category="laptop"
[  0.002s] search_start   params={"category

* A (без retry) первая же ошибка разбора завершает обработку - 1 попытка, поиск не вызывался
* B (с retry) первая попытка - некорректный JSON, вторая - корректный; поиск вызван ровно один раз и только
  после parse_ok -> validate_ok
* C (с retry, все ответы плохие) три попытки (лимит, включая первую), затем failed и понятное сообщение
  пользователю; поиск не вызывался ни разу

Retry охватывает только генерацию + парсинг, поэтому повторы не приводят к повторному обращению к SQLite

## Часть 5. Уточняющие запросы

StateDialog хранит последний валидный ProductQuery в переменной программы и передает его в промпт вместе с новой репликой.
HistoryDialog вместо этого передает всю историю диалога (MessagesPlaceholder)

In [28]:
from src.dialog import StateDialog, HistoryDialog

DIALOGS = {
    "Диалог 1": [
        ("Покажи ноутбуки до 80 тысяч", ("laptop", 80000, "price_asc")),
        ("А сначала дорогие?", ("laptop", 80000, "price_desc")),
        ("А что есть среди смартфонов?", ("smartphone", 80000, "price_desc")),
    ],
    "Диалог 2": [
        ("Хочу посмотреть планшеты", ("tablet", None, "price_asc")),
        ("Только не дороже 40 тысяч", ("tablet", 40000, "price_asc")),
        ("А если ноутбуки, но до 70?", ("laptop", 70000, "price_asc")),
        ("Покажи от дорогих к дешевым", ("laptop", 70000, "price_desc")),
    ],
}
RESULTS["part5"] = {}
for name, turns in DIALOGS.items():
    sd, hd = StateDialog(llm), HistoryDialog(llm)
    rows = []
    for i, (text, exp) in enumerate(turns, 1):
        rs, rh = sd.send(text), hd.send(text)
        rows.append((i, text, f"{exp[0]}, {exp[1]}, {exp[2]}",
                     params(rs.query), check(rs.query, exp), (rs.usage or {}).get("input_tokens"),
                     params(rh.query), check(rh.query, exp), (rh.usage or {}).get("input_tokens")))
    display(Markdown(f"**{name}**"))
    md_table(["#", "Реплика", "Ожидается", "State: параметры", "State", "State in_tok",
              "History: параметры", "History", "History in_tok"], rows)
    print("Найдено на последнем шаге (state):", format_products(rs.products))
    RESULTS["part5"][name] = [list(r) for r in rows]

**Диалог 1**

| # | Реплика | Ожидается | State: параметры | State | State in_tok | History: параметры | History | History in_tok |
|---|---|---|---|---|---|---|---|---|
| 1 | Покажи ноутбуки до 80 тысяч | laptop, 80000, price_asc | laptop, 80000, price_asc | Верно | 739 | laptop, 80000, price_asc | Верно | 707 |
| 2 | А сначала дорогие? | laptop, 80000, price_desc | laptop, 80000, price_desc | Верно | 748 | laptop, 80000, price_desc | Верно | 746 |
| 3 | А что есть среди смартфонов? | smartphone, 80000, price_desc | smartphone, None, price_asc | Ошибка: max_price, sort_by | 751 | smartphone, None, price_asc | Ошибка: max_price, sort_by | 788 |

Найдено на последнем шаге (state): Xiaomi REDMI Note 15 Pro (24 299 ₽); Samsung Galaxy A57 (41 999 ₽); POCO X8 Pro Max (58 799 ₽); HONOR 600 (59 999 ₽); Apple iPhone 17 (88 790 ₽)


**Диалог 2**

| # | Реплика | Ожидается | State: параметры | State | State in_tok | History: параметры | History | History in_tok |
|---|---|---|---|---|---|---|---|---|
| 1 | Хочу посмотреть планшеты | tablet, None, price_asc | tablet, None, price_asc | Верно | 736 | tablet, None, price_asc | Верно | 704 |
| 2 | Только не дороже 40 тысяч | tablet, 40000, price_asc | tablet, 40000, price_asc | Верно | 747 | tablet, 40000, price_asc | Верно | 742 |
| 3 | А если ноутбуки, но до 70? | laptop, 70000, price_asc | laptop, 70000, price_asc | Верно | 754 | laptop, 70000, price_asc | Верно | 787 |
| 4 | Покажи от дорогих к дешевым | laptop, 70000, price_desc | laptop, 70000, price_desc | Верно | 753 | laptop, 70000, price_desc | Верно | 831 |

Найдено на последнем шаге (state): Tecno Megabook K15SDA (55 299 ₽); DEXP Atlas A15-I3W400 (44 299 ₽)


Оба варианта верно обработали 6 из 7 реплик, включая уточнения «А сначала дорогие?» и «Только не дороже 40 тысяч». На реплике «А что есть среди смартфонов?» оба варианта сбросили цену и сортировку (smartphone, None, price_asc), то есть модель поняла смену категории как новый поиск. Ожидаемой мы считали интерпретацию «то же, но среди смартфонов» (сохранить 80000 и price_desc). Это неоднозначность самой реплики и правила в промпте «если пользователь начинает новый поиск - верни новые параметры»; исправляется уточнением правила (смена категории не сбрасывает остальные параметры, если пользователь не сказал иначе)

Входные токены: в варианте с состоянием размер промпта почти постоянный (736-754 токена), так как передается только текущий ProductQuery. В варианте с историей он растет с каждой репликой (704 -> 831 за 4 шага), потому что в промпт добавляются все прошлые реплики и ответы. На длинных диалогах хранение состояния вне модели дешевле и не зависит от длины истории

### Исправление ошибки уточнения: правила версии 2

В прогоне выше на реплике «А что есть среди смартфонов?» оба варианта сбросили цену и сортировку. Причина в промпте - общие правила говорят «если цены нет -> null» и
«ничего не сказано -> price_asc», а правило диалога разрешает «новый поиск». Модель применила значения по умолчанию к уточнению

Поэтому в src/chains.py добавлены промпты state_prompt_v2 и history_prompt_v2 с блоком FOLLOWUP_RULES_V2:

значения по умолчанию относятся только к первому запросу, смена категории - уточнение (цена и сортировка сохраняются),
сброс только по явной просьбе. Ниже оба варианта диалога прогоняются с правилами v1 и v2. Добавлен диалог 3, который
проверяет, что исправление не мешает явному сбросу цены и новому поиску

In [29]:

from src.chains import FOLLOWUP_RULES_V2, state_prompt, history_prompt, state_prompt_v2, history_prompt_v2

print(FOLLOWUP_RULES_V2.strip())


Правила для уточняющих реплик (важнее правил по умолчанию выше):
- Значения по умолчанию (max_price = null, sort_by = price_asc) применяются только к первому запросу диалога.
  В уточнении параметр, о котором пользователь не говорит, сохраняет текущее значение.
- Смена категории («а что среди смартфонов?», «а если планшеты?», «а ноутбуки?») — это уточнение, а не новый поиск:
  меняй только category, а max_price и sort_by оставляй прежними.
- Сбрасывай ограничение цены только по явной просьбе: «без ограничения цены», «любой цены», «неважно сколько стоит».
- Начинай поиск с нуля только если пользователь прямо об этом говорит: «новый поиск», «начнем заново», «забудь прошлое».

Примеры (текущие параметры + реплика → результат):
- laptop, 80000, price_desc + «А что есть среди смартфонов?» → smartphone, 80000, price_desc
- tablet, 40000, price_asc + «А сначала дорогие?» → tablet, 40000, price_desc
- smartphone, 30000, price_desc + «А без ограничения по цене?» → smartphone, null, price_desc
-

In [30]:

DIALOGS_V2 = dict(DIALOGS)
DIALOGS_V2["Диалог 3"] = [
    ("Хочу смартфон подешевле, до 35 тысяч", ("smartphone", 35000, "price_asc")),
    ("А среди планшетов?", ("tablet", 35000, "price_asc")),
    ("Цена неважна", ("tablet", None, "price_asc")),
    ("Начнем заново: ноутбуки, сначала дорогие", ("laptop", None, "price_desc")),
]
PROMPTS = {
    "v1": (state_prompt, history_prompt),
    "v2": (state_prompt_v2, history_prompt_v2),
}

summary, RESULTS["part5_v2"] = [], {}
for name, turns in DIALOGS_V2.items():
    rows = []
    runs = {}
    for ver, (sp, hp) in PROMPTS.items():
        sd, hd = StateDialog(llm, prompt=sp), HistoryDialog(llm, prompt=hp)
        runs[ver] = [(sd.send(text), hd.send(text)) for text, _ in turns]
    for k, (text, exp) in enumerate(turns):
        row = [k + 1, text, f"{exp[0]}, {exp[1]}, {exp[2]}"]
        for ver in PROMPTS:
            rs, rh = runs[ver][k]
            row += [params(rs.query), check(rs.query, exp), params(rh.query), check(rh.query, exp)]
        rows.append(row)
    display(Markdown(f"**{name}**"))
    md_table(["#", "Реплика", "Ожидается",
              "v1 State", "v1 State: проверка", "v1 History", "v1 History: проверка",
              "v2 State", "v2 State: проверка", "v2 History", "v2 History: проверка"], rows)
    RESULTS["part5_v2"][name] = rows
    for ver in PROMPTS:
        for kind, j in (("State", 0), ("History", 1)):
            ok = sum(check(runs[ver][k][j].query, turns[k][1]) == "Верно" for k in range(len(turns)))
            tok = [(runs[ver][k][j].usage or {}).get("input_tokens") for k in range(len(turns))]
            summary.append((name, ver, kind, f"{ok}/{len(turns)}", " → ".join(map(str, tok))))

md_table(["Диалог", "Правила", "Вариант", "Верно", "input_tokens по шагам"], summary)
RESULTS["part5_v2_summary"] = [list(r) for r in summary]


**Диалог 1**

| # | Реплика | Ожидается | v1 State | v1 State: проверка | v1 History | v1 History: проверка | v2 State | v2 State: проверка | v2 History | v2 History: проверка |
|---|---|---|---|---|---|---|---|---|---|---|
| 1 | Покажи ноутбуки до 80 тысяч | laptop, 80000, price_asc | laptop, 80000, price_asc | Верно | laptop, 80000, price_asc | Верно | laptop, 80000, price_asc | Верно | laptop, 80000, price_asc | Верно |
| 2 | А сначала дорогие? | laptop, 80000, price_desc | laptop, 80000, price_desc | Верно | laptop, 80000, price_desc | Верно | laptop, 80000, price_desc | Верно | laptop, 80000, price_desc | Верно |
| 3 | А что есть среди смартфонов? | smartphone, 80000, price_desc | smartphone, None, price_asc | Ошибка: max_price, sort_by | smartphone, None, price_asc | Ошибка: max_price, sort_by | smartphone, 80000, price_desc | Верно | smartphone, 80000, price_desc | Верно |

**Диалог 2**

| # | Реплика | Ожидается | v1 State | v1 State: проверка | v1 History | v1 History: проверка | v2 State | v2 State: проверка | v2 History | v2 History: проверка |
|---|---|---|---|---|---|---|---|---|---|---|
| 1 | Хочу посмотреть планшеты | tablet, None, price_asc | tablet, None, price_asc | Верно | tablet, None, price_asc | Верно | tablet, None, price_asc | Верно | tablet, None, price_asc | Верно |
| 2 | Только не дороже 40 тысяч | tablet, 40000, price_asc | tablet, 40000, price_asc | Верно | tablet, 40000, price_asc | Верно | tablet, 40000, price_asc | Верно | tablet, 40000, price_asc | Верно |
| 3 | А если ноутбуки, но до 70? | laptop, 70000, price_asc | laptop, 70000, price_asc | Верно | laptop, 70000, price_asc | Верно | laptop, 70000, price_asc | Верно | laptop, 70000, price_asc | Верно |
| 4 | Покажи от дорогих к дешевым | laptop, 70000, price_desc | laptop, 70000, price_desc | Верно | laptop, 70000, price_desc | Верно | laptop, 70000, price_desc | Верно | laptop, 70000, price_desc | Верно |

**Диалог 3**

| # | Реплика | Ожидается | v1 State | v1 State: проверка | v1 History | v1 History: проверка | v2 State | v2 State: проверка | v2 History | v2 History: проверка |
|---|---|---|---|---|---|---|---|---|---|---|
| 1 | Хочу смартфон подешевле, до 35 тысяч | smartphone, 35000, price_asc | smartphone, 35000, price_asc | Верно | smartphone, 35000, price_asc | Верно | smartphone, 35000, price_asc | Верно | smartphone, 35000, price_asc | Верно |
| 2 | А среди планшетов? | tablet, 35000, price_asc | tablet, 35000, price_asc | Верно | tablet, 35000, price_asc | Верно | tablet, 35000, price_asc | Верно | tablet, 35000, price_asc | Верно |
| 3 | Цена неважна | tablet, None, price_asc | tablet, None, price_asc | Верно | tablet, None, price_asc | Верно | tablet, None, price_asc | Верно | tablet, None, price_asc | Верно |
| 4 | Начнем заново: ноутбуки, сначала дорогие | laptop, None, price_desc | laptop, None, price_desc | Верно | laptop, None, price_desc | Верно | laptop, None, price_desc | Верно | laptop, None, price_desc | Верно |

| Диалог | Правила | Вариант | Верно | input_tokens по шагам |
|---|---|---|---|---|
| Диалог 1 | v1 | State | 2/3 | 739 → 748 → 751 |
| Диалог 1 | v1 | History | 2/3 | 707 → 746 → 788 |
| Диалог 1 | v2 | State | 3/3 | 1063 → 1072 → 1075 |
| Диалог 1 | v2 | History | 3/3 | 1031 → 1070 → 1112 |
| Диалог 2 | v1 | State | 4/4 | 736 → 747 → 754 → 753 |
| Диалог 2 | v1 | History | 4/4 | 704 → 742 → 787 → 831 |
| Диалог 2 | v2 | State | 4/4 | 1060 → 1071 → 1078 → 1077 |
| Диалог 2 | v2 | History | 4/4 | 1028 → 1066 → 1111 → 1155 |
| Диалог 3 | v1 | State | 4/4 | 743 → 749 → 747 → 751 |
| Диалог 3 | v1 | History | 4/4 | 711 → 751 → 789 → 831 |
| Диалог 3 | v2 | State | 4/4 | 1067 → 1073 → 1071 → 1075 |
| Диалог 3 | v2 | History | 4/4 | 1035 → 1075 → 1113 → 1155 |

* Ошибка исправлена. С правилами v2 реплика «А что есть среди смартфонов?» обрабатывается как уточнение
  (smartphone, 80000, price_desc) в обоих вариантах: диалог 1 - 3/3 вместо 2/3
* Диалог 2 - 4/4 в обоих версиях. Диалог 3 проверяет обратную сторону: явный сброс цены
  («Цена неважна» -> null) и новый поиск («Начнем заново: ноутбуки, сначала дорогие») v2 обработал верно, 4/4.
* Оговорка - реплика диалога 1 совпадает с первым примером в правилах v2, поэтому этот шаг отчасти проверяет
  запоминание примера. При этом похожая смена категории в диалоге 3 («А среди планшетов?») верно обработана в обеих
  версиях: ошибка v1 проявлялась не на любой смене категории, а на конкретной формулировке.
* Правила v2 удлиняют системный промпт примерно на 324 входных токена на каждый вызов. Общий вывод части 5 сохраняется: в варианте с состоянием размер промпта почти постоянный, а в варианте с историей растет с каждой репликой

In [31]:
out = ROOT / "report" / "results.json"
out.write_text(json.dumps(RESULTS, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Сохранено:", out.relative_to(ROOT))

Сохранено: report\results.json
